# Ginem API — Thesis Eval Results Viewer

Visualize outputs from `CORE/script/eval/output/` produced by:

| Runner | Typical files |
|---|---|
| **E2E** (`run-e2e-eval.ts`) | `e2e-results.csv`, `summary-by-model.*`, `system-integration-summary.*` |
| **LLM isolated** (`run-llm-eval.ts`) | `llm-results.csv`, `summary-by-model.*`, `summary-by-category.*` |
| **Integration** (`run-integration.ts`) | `integration-results.csv`, `integration-summary.*` |

## How to use in Google Colab

1. Open this notebook in [Google Colab](https://colab.research.google.com/) (**File → Upload notebook**).
2. Zip your local `CORE/script/eval/output` folder (or a single run under `processed/<runId>`).
3. Run the setup cells, then upload the zip **or** mount Google Drive and set `RESULTS_ROOT`.
4. Pick a run id and explore tables + charts.

> Tip: You can also run this notebook locally with Jupyter if `RESULTS_ROOT` points at `CORE/script/eval/output`.

## 1. Setup

In [ ]:
# Optional: uncomment if you need extra packages in Colab
# %pip install -q pandas matplotlib seaborn

from __future__ import annotations

import json
import shutil
import zipfile
from pathlib import Path
from typing import Any

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 80)

WORK = Path("/content/ginem_eval")
WORK.mkdir(parents=True, exist_ok=True)
print("Workspace:", WORK)

## 2. Load results

Choose **one** method below, then run the discovery cell.

### Option A — Upload a zip (Colab)

In [ ]:
USE_UPLOAD = True  # set False if using Drive / local path instead

if USE_UPLOAD:
    try:
        from google.colab import files  # type: ignore
    except ImportError as e:
        raise SystemExit(
            "google.colab is only available in Colab. "
            "Set USE_UPLOAD=False and use Option B/C."
        ) from e

    print("Upload a zip of eval/output (or processed/<runId>)…")
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("No file uploaded.")

    zip_name = next(iter(uploaded))
    zip_path = WORK / zip_name
    zip_path.write_bytes(uploaded[zip_name])

    extract_dir = WORK / "uploaded"
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)

    RESULTS_ROOT = extract_dir
    print("Extracted to:", RESULTS_ROOT)
    print("Top-level entries:", [p.name for p in RESULTS_ROOT.iterdir()][:20])
else:
    print("Skipping upload (USE_UPLOAD=False).")

### Option B — Google Drive

In [ ]:
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
    # Update this path to your Drive folder that contains processed/ raw/ logs/
    RESULTS_ROOT = Path("/content/drive/MyDrive/ginem-eval-output")
    print("RESULTS_ROOT =", RESULTS_ROOT)
    assert RESULTS_ROOT.exists(), f"Not found: {RESULTS_ROOT}"

### Option C — Local path (Jupyter / Colab with uploaded folder)

In [ ]:
USE_LOCAL = False

if USE_LOCAL:
    # Example when running beside the repo:
    # RESULTS_ROOT = Path("../script/eval/output")
    RESULTS_ROOT = Path("/content/ginem_eval/uploaded")  # change as needed
    print("RESULTS_ROOT =", RESULTS_ROOT)
    assert RESULTS_ROOT.exists(), f"Not found: {RESULTS_ROOT}"

### Discover runs

In [ ]:
def find_processed_roots(root: Path) -> list[Path]:
    """Find directories that look like processed/<runId>."""
    root = root.resolve()
    candidates: list[Path] = []

    processed = root / "processed"
    if processed.is_dir():
        candidates.extend([p for p in processed.iterdir() if p.is_dir()])

    # Zip may nest: output/processed/<runId>
    for p in root.rglob("processed"):
        if p.is_dir():
            candidates.extend([c for c in p.iterdir() if c.is_dir()])

    # Or the uploaded folder IS a single run
    markers = (
        "e2e-results.csv",
        "llm-results.csv",
        "integration-results.csv",
        "summary.json",
        "integration-summary.json",
    )
    if any((root / m).exists() for m in markers):
        candidates.append(root)

    # unique, sorted by mtime desc
    uniq = {str(c.resolve()): c.resolve() for c in candidates}
    runs = sorted(uniq.values(), key=lambda p: p.stat().st_mtime, reverse=True)
    return runs


def detect_run_kind(run_dir: Path) -> str:
    if (run_dir / "e2e-results.csv").exists() or (run_dir / "e2e-results.json").exists():
        return "e2e"
    if (run_dir / "llm-results.csv").exists():
        return "llm"
    if (run_dir / "integration-results.csv").exists():
        return "integration"
    if (run_dir / "system-integration-summary.json").exists():
        return "e2e"
    return "unknown"


if "RESULTS_ROOT" not in globals():
    raise SystemExit("Set RESULTS_ROOT via Option A, B, or C first.")

RUNS = find_processed_roots(Path(RESULTS_ROOT))
if not RUNS:
    raise SystemExit(
        f"No processed runs found under {RESULTS_ROOT}. "
        "Expected processed/<runId>/ with CSV/JSON artifacts."
    )

run_table = pd.DataFrame(
    [
        {
            "runId": r.name,
            "kind": detect_run_kind(r),
            "path": str(r),
            "files": ", ".join(sorted(p.name for p in r.iterdir() if p.is_file())[:12]),
        }
        for r in RUNS
    ]
)
display(run_table)

# Default: newest run
RUN_DIR = RUNS[0]
RUN_KIND = detect_run_kind(RUN_DIR)
print(f"\nSelected RUN_DIR = {RUN_DIR}")
print(f"Detected kind     = {RUN_KIND}")

In [ ]:
# Optionally pick another run by name
SELECTED_RUN_ID = None  # e.g. "e2e-2026-08-02T04-21-42-770Z"

if SELECTED_RUN_ID:
    matches = [r for r in RUNS if r.name == SELECTED_RUN_ID]
    if not matches:
        raise SystemExit(f"Run id not found: {SELECTED_RUN_ID}")
    RUN_DIR = matches[0]
    RUN_KIND = detect_run_kind(RUN_DIR)

print("Using:", RUN_DIR.name, f"({RUN_KIND})")

## 3. Helpers — load CSV / JSON

In [ ]:
def read_csv_if_exists(path: Path) -> pd.DataFrame | None:
    if path.exists():
        return pd.read_csv(path)
    return None


def read_json_if_exists(path: Path) -> Any | None:
    if path.exists():
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)
    return None


def load_run(run_dir: Path) -> dict[str, Any]:
    kind = detect_run_kind(run_dir)
    data: dict[str, Any] = {"kind": kind, "run_dir": run_dir}

    # Case-level results
    for name in ("e2e-results.csv", "llm-results.csv", "integration-results.csv"):
        df = read_csv_if_exists(run_dir / name)
        if df is not None:
            data["results"] = df
            data["results_file"] = name
            break

    # Summaries
    data["by_model"] = read_csv_if_exists(run_dir / "summary-by-model.csv")
    data["by_category"] = read_csv_if_exists(run_dir / "summary-by-category.csv")
    data["system_summary"] = read_csv_if_exists(run_dir / "system-integration-summary.csv")
    data["integration_summary"] = read_csv_if_exists(run_dir / "integration-summary.csv")

    data["summary_json"] = read_json_if_exists(run_dir / "summary.json")
    data["system_summary_json"] = read_json_if_exists(
        run_dir / "system-integration-summary.json"
    )
    data["integration_summary_json"] = read_json_if_exists(
        run_dir / "integration-summary.json"
    )

    # Raw sibling (optional)
    raw_guess = run_dir.parent.parent / "raw" / run_dir.name
    if not raw_guess.exists():
        found = list(Path(RESULTS_ROOT).rglob(run_dir.name))
        for p in found:
            if p.name != run_dir.name or "raw" not in str(p):
                continue
            if any(
                (p / name).exists()
                for name in (
                    "e2e-results.json",
                    "llm-results.json",
                    "integration-results.json",
                )
            ):
                raw_guess = p
                break
    data["raw_dir"] = raw_guess if raw_guess.exists() else None

    # Errors log
    log_guess = run_dir.parent.parent / "logs" / run_dir.name / "errors.log"
    if not log_guess.exists():
        alt = list(Path(RESULTS_ROOT).rglob("errors.log"))
        log_guess = next(
            (p for p in alt if run_dir.name in str(p)),
            Path(""),
        )
    data["errors_log"] = log_guess if log_guess and Path(log_guess).exists() else None

    return data


bundle = load_run(RUN_DIR)
print("Kind:", bundle["kind"])
print("Results file:", bundle.get("results_file"))
print("Rows:", len(bundle["results"]) if bundle.get("results") is not None else 0)
print("Raw dir:", bundle.get("raw_dir"))
print("Errors log:", bundle.get("errors_log"))

## 4. Overview KPIs

In [ ]:
results: pd.DataFrame | None = bundle.get("results")
if results is None:
    raise SystemExit("No case-level results CSV found in this run.")

display(results.head(10))
print("\nColumns:", list(results.columns))
print("Shape:", results.shape)

kpi = {}
kpi["n_cases"] = len(results)

if "toolNameCorrect" in results.columns:
    kpi["tool_accuracy_pct"] = 100 * results["toolNameCorrect"].astype(bool).mean()
if "behaviorMatch" in results.columns:
    kpi["behavior_match_pct"] = 100 * results["behaviorMatch"].astype(bool).mean()
if "functionalSuccess" in results.columns:
    kpi["functional_success_pct"] = 100 * results["functionalSuccess"].astype(bool).mean()
if "integrationSuccess" in results.columns:
    kpi["integration_success_pct"] = 100 * results["integrationSuccess"].astype(bool).mean()
if "mqttSuccess" in results.columns:
    kpi["mqtt_success_pct"] = 100 * results["mqttSuccess"].astype(bool).mean()
if "latencyMs" in results.columns:
    kpi["avg_latency_ms"] = float(results["latencyMs"].mean())
    kpi["p95_latency_ms"] = float(results["latencyMs"].quantile(0.95))
if "estimatedCostUsd" in results.columns:
    kpi["total_estimated_cost_usd"] = float(results["estimatedCostUsd"].sum())
if "totalTokens" in results.columns:
    kpi["total_tokens"] = float(results["totalTokens"].sum())
if "error" in results.columns:
    kpi["error_rows"] = int(results["error"].notna().sum() & (results["error"].astype(str) != "").sum())

kpi_df = pd.DataFrame([kpi]).T.reset_index()
kpi_df.columns = ["metric", "value"]
display(kpi_df)

if bundle.get("system_summary") is not None:
    print("\nSystem integration summary CSV:")
    display(bundle["system_summary"])
if bundle.get("integration_summary") is not None:
    print("\nIntegration summary CSV:")
    display(bundle["integration_summary"])
if bundle.get("summary_json") is not None:
    print("\nsummary.json keys:", list(bundle["summary_json"].keys()) if isinstance(bundle["summary_json"], dict) else type(bundle["summary_json"]))

## 5. Summary by model

In [ ]:
by_model = bundle.get("by_model")

if by_model is None and results is not None and "modelId" in results.columns:
    # Derive a lightweight summary if CSV summary is missing
    aggs = {"latencyMs": "mean"}
    if "toolNameCorrect" in results.columns:
        aggs["toolNameCorrect"] = "mean"
    if "behaviorMatch" in results.columns:
        aggs["behaviorMatch"] = "mean"
    if "functionalSuccess" in results.columns:
        aggs["functionalSuccess"] = "mean"
    if "mqttSuccess" in results.columns:
        aggs["mqttSuccess"] = "mean"
    if "estimatedCostUsd" in results.columns:
        aggs["estimatedCostUsd"] = "sum"
    by_model = results.groupby("modelId", as_index=False).agg(aggs)
    for col in by_model.columns:
        if col.endswith("Correct") or col.endswith("Match") or col.endswith("Success"):
            by_model[col] = by_model[col] * 100

if by_model is not None:
    display(by_model)

    metric_candidates = [
        c
        for c in [
            "toolAccuracyPct",
            "parameterAccuracyPct",
            "behaviorMatchRatePct",
            "averageLatencyMs",
            "estimatedCostUsd",
            "toolNameCorrect",
            "behaviorMatch",
            "functionalSuccess",
            "mqttSuccess",
            "latencyMs",
        ]
        if c in by_model.columns
    ]

    id_col = "modelId" if "modelId" in by_model.columns else by_model.columns[0]
    plot_metrics = metric_candidates[:4]
    if plot_metrics:
        n = len(plot_metrics)
        fig, axes = plt.subplots(1, n, figsize=(4.2 * n, 4))
        if n == 1:
            axes = [axes]
        for ax, col in zip(axes, plot_metrics):
            sns.barplot(data=by_model, x=id_col, y=col, ax=ax, hue=id_col, legend=False)
            ax.set_title(col)
            ax.tick_params(axis="x", rotation=25)
        fig.suptitle("Summary by model")
        fig.tight_layout()
        plt.show()
else:
    print("No per-model summary available for this run.")

## 6. Summary by category

In [ ]:
by_cat = bundle.get("by_category")

if by_cat is None and results is not None and "category" in results.columns:
    aggs = {}
    if "toolNameCorrect" in results.columns:
        aggs["toolNameCorrect"] = "mean"
    if "behaviorMatch" in results.columns:
        aggs["behaviorMatch"] = "mean"
    if "latencyMs" in results.columns:
        aggs["latencyMs"] = "mean"
    if "functionalSuccess" in results.columns:
        aggs["functionalSuccess"] = "mean"
    if aggs:
        by_cat = results.groupby("category", as_index=False).agg(aggs)
        for col in list(by_cat.columns):
            if col in ("toolNameCorrect", "behaviorMatch", "functionalSuccess"):
                by_cat[col] = by_cat[col] * 100

if by_cat is not None:
    display(by_cat)
    cat_col = "category" if "category" in by_cat.columns else by_cat.columns[0]
    y_candidates = [
        c
        for c in [
            "toolAccuracyPct",
            "behaviorMatchRatePct",
            "parameterAccuracyPct",
            "averageLatencyMs",
            "toolNameCorrect",
            "behaviorMatch",
            "functionalSuccess",
            "latencyMs",
        ]
        if c in by_cat.columns
    ]
    if y_candidates:
        fig, axes = plt.subplots(1, min(3, len(y_candidates)), figsize=(12, 4))
        if min(3, len(y_candidates)) == 1:
            axes = [axes]
        for ax, col in zip(axes, y_candidates[:3]):
            sns.barplot(data=by_cat, x=cat_col, y=col, ax=ax, hue=cat_col, legend=False)
            ax.set_title(col)
            ax.tick_params(axis="x", rotation=25)
        fig.suptitle("Summary by category")
        fig.tight_layout()
        plt.show()
else:
    print("No per-category summary available for this run.")

## 7. Latency distribution

In [ ]:
if "latencyMs" not in results.columns:
    print("No latencyMs column.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(results["latencyMs"], bins=20, ax=axes[0], kde=True)
    axes[0].set_title("Latency histogram (ms)")
    axes[0].set_xlabel("latencyMs")

    if "modelId" in results.columns:
        sns.boxplot(data=results, x="modelId", y="latencyMs", ax=axes[1])
        axes[1].tick_params(axis="x", rotation=25)
        axes[1].set_title("Latency by model")
    elif "category" in results.columns:
        sns.boxplot(data=results, x="category", y="latencyMs", ax=axes[1])
        axes[1].tick_params(axis="x", rotation=25)
        axes[1].set_title("Latency by category")
    else:
        axes[1].axis("off")

    fig.tight_layout()
    plt.show()

    display(results["latencyMs"].describe().to_frame("latencyMs"))

## 8. Success / failure matrix

In [ ]:
bool_cols = [
    c
    for c in [
        "toolNameCorrect",
        "toolCallValid",
        "behaviorMatch",
        "functionalSuccess",
        "integrationSuccess",
        "mqttSuccess",
    ]
    if c in results.columns
]

if "category" in results.columns and bool_cols:
    for col in bool_cols:
        pivot = (
            results.assign(**{col: results[col].astype(bool)})
            .groupby("category")[col]
            .mean()
            .mul(100)
            .rename(f"{col}_pct")
        )
        display(pivot.to_frame())

    # Heatmap of mean success rates
    heat = (
        results.assign(**{c: results[c].astype(bool) for c in bool_cols})
        .groupby("category")[bool_cols]
        .mean()
        .mul(100)
    )
    fig, ax = plt.subplots(figsize=(1.6 * len(bool_cols) + 2, 4))
    sns.heatmap(heat, annot=True, fmt=".1f", cmap="YlGnBu", ax=ax, vmin=0, vmax=100)
    ax.set_title("Success rate (%) by category")
    plt.tight_layout()
    plt.show()
else:
    print("Need category + boolean success columns for this section.")

## 9. Failures & errors

In [ ]:
fail_mask = pd.Series(False, index=results.index)
for col in ("toolNameCorrect", "behaviorMatch", "functionalSuccess", "mqttSuccess"):
    if col in results.columns:
        fail_mask = fail_mask | (~results[col].astype(bool))

if "error" in results.columns:
    err_series = results["error"].fillna("").astype(str).str.strip()
    fail_mask = fail_mask | (err_series != "")

failures = results.loc[fail_mask].copy()
print(f"Failure / error rows: {len(failures)} / {len(results)}")
show_cols = [
    c
    for c in [
        "caseId",
        "category",
        "modelId",
        "command",
        "rawToolNames",
        "toolNameCorrect",
        "behaviorMatch",
        "functionalSuccess",
        "mqttSuccess",
        "latencyMs",
        "error",
        "replyPreview",
    ]
    if c in failures.columns
]
display(failures[show_cols].head(50))

if bundle.get("errors_log") is not None:
    log_path = Path(bundle["errors_log"])
    text = log_path.read_text(encoding="utf-8", errors="replace")
    print(f"\n=== {log_path} (tail) ===")
    print("\n".join(text.strip().splitlines()[-40:]))

## 10. Tokens & cost (isolated LLM runs)

In [ ]:
token_cols = [c for c in ("inputTokens", "outputTokens", "totalTokens", "estimatedCostUsd") if c in results.columns]

if not token_cols:
    print("No token/cost columns in this run (common for E2E ChatService traces).")
elif results[token_cols].fillna(0).sum().sum() == 0:
    print("Token/cost columns are present but all zeros.")
    display(results[token_cols].describe())
else:
    if "modelId" in results.columns:
        cost = results.groupby("modelId")[token_cols].sum(numeric_only=True)
        display(cost)
        if "estimatedCostUsd" in cost.columns:
            fig, ax = plt.subplots(figsize=(6, 4))
            cost["estimatedCostUsd"].plot(kind="bar", ax=ax)
            ax.set_ylabel("USD")
            ax.set_title("Estimated API cost by model")
            plt.tight_layout()
            plt.show()
    else:
        display(results[token_cols].sum().to_frame("total"))

## 11. Export filtered table (optional)

In [ ]:
export_path = WORK / f"{RUN_DIR.name}_failures.csv"
failures.to_csv(export_path, index=False)
print("Wrote", export_path)

try:
    from google.colab import files  # type: ignore

    files.download(str(export_path))
except Exception:
    print("Download skipped (not in Colab or download unavailable).")

---

### Scoring reference

| Metric | Formula |
|---|---|
| Tool accuracy | correct tool selections / total tests × 100% |
| Parameter accuracy | correct expected parameters / total expected parameters × 100% |
| Latency | response timestamp − request timestamp |
| API cost | inputTokens/1e6 × inputPrice + outputTokens/1e6 × outputPrice |

See also `CORE/script/README.md` and `CORE/script/eval/README.md`.